# Structs

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Agrupar valores relacionados numa struct, dar um nome a ela com typedef, e usar arrays de structs.
- Passar structs para funções por valor ou por um ponteiro com ->, e explicar o que copiar uma struct copia.
- Dar nome a um conjunto fixo de opções com um enum e fazer switch nele.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma struct agrupa valores
2. Atribuir copia todos os campos
3. Structs e funções
4. Arrays de structs, e arrays dentro de structs
5. enum: dando nome a um conjunto de opções
6. sizeof e preenchimento
7. Comparando structs

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Uma struct agrupa valores

Uma **struct** agrupa valores de tipos diferentes com um nome só, como uma classe do Java só com campos. O `typedef` dá um nome curto ao tipo, para você escrever `Point` em vez de `struct Point`. Crie uma com os valores entre chaves, em ordem, ou pelo nome do campo (`.y = 4`), e alcance os campos com um ponto.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int main(void) {
    Point a = {3, 4};
    Point b = {.y = 10, .x = -1};
    a.x = a.x + 1;
    printf("a = (%d, %d), b = (%d, %d)\n", a.x, a.y, b.x, b.y);
    return 0;
}

🧪 **Experimente:** acrescente um campo `char label` à struct, dê um valor a ele nos dois pontos e imprima.

## 2. Atribuir copia todos os campos

`Point b = a;` **copia todos os campos**: `b` é uma struct separada, e mudá-la não mexe em `a`. É diferente dos objetos do Java, em que atribuir compartilha um objeto. Para trabalhar no original, use um ponteiro. Os slides mostram os dois casos.

#### 🎞️ Cópias e ponteiros de struct

![Point b = a copia os campos, então mudar b não mexe em a; um ponteiro p = &a muda o próprio a](struct-copy-steps.pt-br.svg)

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int main(void) {
    Point a = {3, 4};
    Point b = a;          // uma cópia
    b.x = 9;
    Point *p = &a;        // um ponteiro para o original
    p->x = 7;             // p->x quer dizer (*p).x
    printf("a = (%d, %d), b = (%d, %d)\n", a.x, a.y, b.x, b.y);
    return 0;
}

🧪 **Experimente:** preveja os dois pontos antes de rodar. Depois escreva `(*p).y = 0;` e confira que faz o mesmo que `p->y = 0;`.

## 3. Structs e funções

Passar uma struct para uma função **a copia**, o que serve para ler uma struct pequena. Quando a função precisa **mudar** a struct de quem chamou, ou a struct é grande, passe um ponteiro (`Rect *r`) e use `r->campo`. Acrescente `const` quando a função só lê pelo ponteiro.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

void move_copy(Point p, int dx) {     // recebe uma cópia
    p.x += dx;
}

void move(Point *p, int dx) {         // recebe o endereço
    p->x += dx;
}

int main(void) {
    Point a = {0, 0};
    move_copy(a, 5);
    printf("depois de move_copy: %d\n", a.x);
    move(&a, 5);
    printf("depois de move: %d\n", a.x);
    return 0;
}

🧪 **Experimente:** faça `move_copy` **devolver** o ponto movido (`Point move_copy(...)`) e use como `a = move_copy(a, 5);`: outro jeito de tirar o resultado de lá.

### ✏️ Exercício: A área de um retângulo

Devolva a área de um Rect passado por valor: a função recebe uma cópia, o que basta para ler.

In [ ]:
typedef struct {
    int w;
    int h;
} Rect;

int rect_area(Rect r) {
    // TODO: multiplique os campos
    return 0;
}

int main(void) {
    Rect r = {3, 4};
    check_int("3 por 4", rect_area(r), 12);
    check_int("uma linha", rect_area((Rect) {5, 0}), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
typedef struct {
    int w;
    int h;
} Rect;

int rect_area(Rect r) {
    return r.w * r.h;
}

int main(void) {
    Rect r = {3, 4};
    check_int("3 por 4", rect_area(r), 12);
    check_int("uma linha", rect_area((Rect) {5, 0}), 0);
    return 0;
}
```

</details>

### ✏️ Exercício: Aumente um retângulo

Multiplique os dois lados por k, mudando o Rect de quem chamou. Isso pede um ponteiro e `->`.

In [ ]:
typedef struct {
    int w;
    int h;
} Rect;

void scale(Rect *r, int k) {
    // TODO: mude r->w e r->h
}

int main(void) {
    Rect r = {3, 4};
    scale(&r, 2);
    check_int("w dobrou", r.w, 6);
    check_int("h dobrou", r.h, 8);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
typedef struct {
    int w;
    int h;
} Rect;

void scale(Rect *r, int k) {
    r->w *= k;
    r->h *= k;
}

int main(void) {
    Rect r = {3, 4};
    scale(&r, 2);
    check_int("w dobrou", r.w, 6);
    check_int("h dobrou", r.h, 8);
    return 0;
}
```

</details>

## 4. Arrays de structs, e arrays dentro de structs

Structs e arrays se combinam dos dois jeitos. Um **array de structs** é uma tabela: `accounts[1].balance`. Uma struct pode guardar um **array**, como um nome de tamanho fixo: `char owner[21]`. Copie texto nele com `snprintf` ou `strcpy` (depois de conferir que cabe), nunca com `=`.

In [ ]:
#include <stdio.h>

typedef struct {
    char name[16];
    int score;
} Player;

int main(void) {
    Player team[3] = {{"Ada", 12}, {"Alan", 30}, {"Grace", 21}};
    snprintf(team[0].name, sizeof team[0].name, "%s", "Ada L.");
    int best = 0;
    for (int i = 1; i < 3; i++) {
        if (team[i].score > team[best].score) {
            best = i;
        }
    }
    for (int i = 0; i < 3; i++) {
        printf("%-8s %3d%s\n", team[i].name, team[i].score, i == best ? "  <- melhor" : "");
    }
    return 0;
}

🧪 **Experimente:** tente `team[0].name = "Bob";` e leia o erro: não dá para atribuir a um array. Depois acrescente um quarto jogador.

## 5. enum: dando nome a um conjunto de opções

Um **enum** dá nomes a um conjunto fixo de opções. Por trás dos nomes há ints (0, 1, 2… por padrão), mas os nomes deixam o código legível e combinam bem com um `switch`.

In [ ]:
#include <stdio.h>

typedef enum { RED, YELLOW, GREEN } Light;

Light next(Light l) {
    switch (l) {
        case RED:
            return GREEN;
        case GREEN:
            return YELLOW;
        case YELLOW:
            return RED;
    }
    return RED;
}

int main(void) {
    const char *names[] = {"vermelho", "amarelo", "verde"};
    Light l = RED;
    for (int i = 0; i < 4; i++) {
        printf("%s (%d)\n", names[l], l);
        l = next(l);
    }
    return 0;
}

🧪 **Experimente:** apague as linhas do `case YELLOW:` e leia o aviso: o switch não trata todos os valores do enum.

## 6. sizeof e preenchimento

O tamanho de uma struct pode ser **maior** que a soma dos campos: o compilador pode acrescentar bytes sem uso (**preenchimento**, ou padding) para cada campo começar num endereço que o processador lê rápido. Ordenar os campos do maior para o menor costuma reduzir esse preenchimento.

In [ ]:
#include <stdio.h>

typedef struct {
    char a;
    double b;
    char c;
} Loose;

typedef struct {
    double b;
    char a;
    char c;
} Tight;

int main(void) {
    printf("campos: %zu bytes\n", sizeof(char) + sizeof(double) + sizeof(char));
    printf("Loose: %zu bytes, Tight: %zu bytes\n", sizeof(Loose), sizeof(Tight));
    return 0;
}

🧪 **Experimente:** acrescente um campo `int` em cada struct, em lugares diferentes, e compare os tamanhos.

## 7. Comparando structs

`==` não funciona em structs: o código não compila. Escreva uma função que compara campo por campo (e com `strcmp` para campos de texto). Não compare structs byte a byte com `memcmp`: os bytes de preenchimento podem ter lixo.

In [ ]:
#include <stdio.h>

typedef struct {
    int x;
    int y;
} Point;

int same_point(Point a, Point b) {
    return a.x == b.x && a.y == b.y;
}

int main(void) {
    Point a = {1, 2};
    Point b = {1, 2};
    Point c = {2, 1};
    printf("a e b: %d, a e c: %d\n", same_point(a, b), same_point(a, c));
    return 0;
}

🧪 **Experimente:** escreva `a == b` no `printf` e leia o erro.

## Em resumo

Uma **struct** agrupa valores de tipos diferentes sob um nome, como uma classe do Java só com campos. `typedef` dá um nome curto ao tipo, e um `enum` dá nome a um conjunto fixo de opções.

```c
#include <stdio.h>

typedef enum { ACTIVE, FROZEN } Status;

typedef struct {
    char owner[21];
    long balance;            // em centavos
    Status status;
} Account;

void deposit(Account *a, long cents) {   // um ponteiro: muda a struct de quem chamou
    if (a->status == ACTIVE) {
        a->balance += cents;               // a->balance é (*a).balance
    }
}

int main(void) {
    Account accounts[2] = {{"Ada", 1000, ACTIVE}, {"Alan", 500, FROZEN}};
    deposit(&accounts[0], 250);
    printf("%s: %ld\n", accounts[0].owner, accounts[0].balance);   // Ada: 1250
    return 0;
}
```

Atribuir ou passar uma struct **copia todos os campos**, inclusive os arrays de dentro, então funções que mudam uma struct, ou trabalham com uma grande, recebem um ponteiro.

**Cuidado:** `==` não funciona com structs. Compare campo por campo.

## Leituras

- [Structs](https://beej.us/guide/bgc/html/split/structs.html) (Beej's Guide to C Programming, chapter 8, 8 min): Declarar uma struct, inicializadores, passar uma struct para uma função (e por que um ponteiro é o comum), o operador seta, a cópia, e por que == não compara structs.
- [Structures and enumerated types](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-structures) (freeCodeCamp, The C Beginner's Handbook, 5 min): Opcional: structs com typedef, e "Enumerated types" logo antes, para o enum usado em Bank Ledger.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Frações** · Fácil
- **Fichas de alunos** · Médio
- **Livro-caixa do banco** · Médio
- **Ponto no polígono** · Difícil